In [35]:
import rasterio as rs
import numpy as np
from rasterio.enums import Resampling

## **Importing & Reading of dataset**

In [36]:
with rs.open("T44RLT_20230225T051801_B04_10m.jp2") as f1,rs.open("T44RLT_20230225T051801_B08_10m.jp2") as f2:
    B04=f1.read(1)
    print(f"Width : {f1.width}")
    print(f"height : {f1.height}")
    print(f"CRS : {f1.crs}")
    print(f"Transform : {f1.transform}")
    print(f"COunt : {f1.count}")
    print(f"Array : {B04}")
    print(f"Max : {B04.max()}")
    print(f"Min : {B04.min()}")
    B08=f2.read(1)
    B08=B08.astype(np.float32)
    B04=B04.astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
      B04=np.where(np.logical_and(B04==0,B08==0),np.nan,B04)
      B08=np.where(np.logical_and(B04==0,B08==0),np.nan,B08)
      NDVI= (B08 - B04) / (B08 + B04)

Width : 10980
height : 10980
CRS : EPSG:32644
Transform : | 10.00, 0.00, 300000.00|
| 0.00,-10.00, 3300000.00|
| 0.00, 0.00, 1.00|
COunt : 1
Array : [[1954 1796 1764 ... 1822 1858 1850]
 [1912 1880 1903 ... 1920 2018 1886]
 [1799 1990 1914 ... 2082 2134 1911]
 ...
 [1348 1398 1361 ... 1486 1503 1480]
 [1350 1353 1334 ... 1511 1500 1496]
 [1330 1333 1364 ... 1511 1479 1481]]
Max : 19007
Min : 0


In [37]:
NDVI.shape

(10980, 10980)

In [38]:
NDVI.mean()

np.float32(0.35115427)

In [39]:
NDVI.max()

np.float32(1.0)

In [40]:
NDVI.min()

np.float32(-1.0)

In [41]:
print(f" Pixels with NDVI>0.4 {(NDVI>0.4).sum()}")
print(f" Pixels with NDVI<0.2 {(NDVI<0.2).sum()}")
print(f" Pixels with NDVI<0 {(NDVI<0).sum()}")

 Pixels with NDVI>0.4 43499288
 Pixels with NDVI<0.2 16008028
 Pixels with NDVI<0 552538


In [42]:
with rs.open("T44RLT_20230225T051801_B12_20m.jp2") as f3:
    B12=f3.read(1)
    print(f"Width : {f3.width}")
    print(f"height : {f3.height}")
    print(f"CRS : {f3.crs}")
    print(f"Transform : {f3.transform}")
    print(f"COunt : {f3.count}")
    print(f"Array : {B12}")
    print(f"Max : {B12.max()}")
    print(f"Min : {B12.min()}")

Width : 5490
height : 5490
CRS : EPSG:32644
Transform : | 20.00, 0.00, 300000.00|
| 0.00,-20.00, 3300000.00|
| 0.00, 0.00, 1.00|
COunt : 1
Array : [[2841 2908 3024 ... 3196 3340 3250]
 [2902 2970 2671 ... 3370 3604 3335]
 [2859 2242 1696 ... 3371 3410 3296]
 ...
 [1510 1524 1600 ... 1951 1793 1670]
 [1487 1494 1521 ... 1847 1673 1644]
 [1491 1512 1510 ... 1841 1698 1650]]
Max : 16408
Min : 934


## **Bug Checked-All pixels valid or not**

In [43]:
NDVI_valid=NDVI.size-np.isnan(NDVI).sum()
print(NDVI_valid)

120560400


In [44]:
print(f" Pixels with NDVI>0.4 {(NDVI>0.4).sum()/NDVI_valid}")
print(f" Pixels with NDVI<0.2 {(NDVI<0.2).sum()/NDVI_valid}")
print(f" Pixels with NDVI<0 {(NDVI<0).sum()/NDVI_valid}")


 Pixels with NDVI>0.4 0.3608090882246575
 Pixels with NDVI<0.2 0.1327801500326807
 Pixels with NDVI<0 0.00458308034810767


## **Reading & Upsampling of B11 & B12**

In [45]:
with rs.open("T44RLT_20230225T051801_B12_20m.jp2") as f12,rs.open("T44RLT_20230225T051801_B11_20m.jp2") as f11:
 upscale_factor=2
 B11=f11.read(
     out_shape=(f11.count,int(f11.height*upscale_factor),int(f11.width*upscale_factor)
 ),
 resampling=Resampling.bilinear
 )
 B12=f12.read(
     out_shape=(f12.count,int(f12.height*upscale_factor),int(f12.width*upscale_factor)
 ),
 resampling=Resampling.bilinear
 )

 print(B11.shape)
 print(B12.shape)



(1, 10980, 10980)
(1, 10980, 10980)


In [46]:
B11=np.squeeze(B11)
B12=np.squeeze(B12)
B11=B11.astype(np.float32)
B12=B12.astype(np.float32)

In [47]:
with np.errstate(divide='ignore', invalid='ignore'):
  B11_masked=np.where(np.logical_and(B11==0,B12==0),np.nan,B11)
  B12_masked=np.where(np.logical_and(B11==0,B12==0),np.nan,B12)
print(B11.shape)
print(B12.shape)

(10980, 10980)
(10980, 10980)


# **NBR**

In [48]:
NBR=(B08-B12_masked)/(B08+B12_masked)
print(NBR.shape)
NBR.size

(10980, 10980)


120560400

In [49]:
NBRmean=NBR.mean()
NBRmax=NBR.max()
NBRmin=NBR.min()
print(NBRmean)
print(NBRmax)
print(NBRmin)

0.21796092
0.66361576
-1.0
